# Retail Demand Forecasting — Exploratory Data Analysis

This notebook explores the Kaggle Store Item Demand Forecasting dataset. Run it from the repository root after placing the competition files in `data/raw/`. The production pipeline and leakage-safe modelling are implemented in `src/retail_forecasting/`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/raw/train.csv') if not Path('data/raw/train.csv').exists() else Path('data/raw/train.csv')
df = pd.read_csv(DATA, parse_dates=['date'])
df.head()

In [ ]:
print('Rows:', len(df))
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Stores:', sorted(df['store'].unique()))
print('Items:', df['item'].nunique())
print('Missing values:\n', df.isna().sum())
print('Duplicate date/store/item rows:', df.duplicated(['date', 'store', 'item']).sum())
df['sales'].describe()

In [ ]:
daily_sales = df.groupby('date', as_index=False)['sales'].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_sales['date'], daily_sales['sales'], linewidth=0.8)
ax.set(title='Total daily sales across stores and items', xlabel='Date', ylabel='Units sold')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
monthly_sales = df.assign(month=df['date'].dt.month).groupby('month')['sales'].mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(monthly_sales.index, monthly_sales.values)
ax.set(title='Average daily item sales by calendar month', xlabel='Month', ylabel='Mean units sold')
ax.grid(axis='y', alpha=0.25)
plt.show()

In [ ]:
# Check that each store-item series has continuous daily records.
coverage = df.groupby(['store', 'item']).agg(
    rows=('date', 'size'), first_date=('date', 'min'), last_date=('date', 'max'), unique_dates=('date', 'nunique')
)
coverage.describe(include='all')

## Interpretation checklist

- Confirm the target is `sales` and the prediction grain is date × store × item.
- Inspect weekly/calendar patterns, outliers, missing records, and whether all series have consistent date coverage.
- Do not randomly shuffle observations for final forecast validation.
- Use the pipeline's final-90-day recursive holdout for model comparison; notebook EDA is descriptive, not a replacement for that evaluation.
- Sales may be a censored proxy for true customer demand if stockouts occurred; this dataset has no inventory or stock availability fields.